# 365 Probabilidades - Dia #110
## Qual a probabilidade de alguém te ajudar se você precisar na rua?

**Tipo:** Comportamental
**Data de publicação:** 2026-10-01
**Ferramenta:** Python
**Decisão analisada:** Dá para contar com quem está por perto?
**Hashtag:** #365Probabilidades #Dia110

---

### 📖 A História

Em 1964, uma mulher chamada Kitty Genovese foi assassinada na porta de casa, em Nova
York. Duas semanas depois, o New York Times publicou que 38 vizinhos tinham acompanhado o
ataque e que nenhum chamou a polícia enquanto ele acontecia.

Aquela manchete empurrou um campo inteiro da psicologia. Virou o "efeito espectador": a
ideia de que quanto mais gente está olhando, menos chance você tem de ser socorrido,
porque cada um supõe que outro vai agir.

Em 2007, três pesquisadores revisaram as transcrições do julgamento e os registros da
época. Não encontraram evidência de 38 testemunhas, nem de que alguém tivesse visto o
ataque inteiro e ficado parado. A manchete estava errada.

O conceito, porém, já tinha saído de casa. Ele é ensinado em escola, repetido em curso
de primeiros socorros, e provavelmente é o que você pensa quando imagina precisar de
ajuda numa rua movimentada.

Em 2019, um grupo de pesquisadores fez a pergunta de outro jeito: em vez de montar um
experimento, foram olhar câmeras de rua.

---

### 📚 O Conceito: duas perguntas que parecem uma só

O efeito espectador clássico é real e está bem medido. Mas ele responde a uma pergunta
específica, e não à que interessa a quem está caído no chão.

**Pergunta 1, a do laboratório:** a chance de **uma pessoa específica** ajudar cai
quando há mais gente em volta? A resposta é sim, e o tamanho está medido.

**Pergunta 2, a da calçada:** a chance de **alguém, qualquer um** ajudar cai quando há
mais gente em volta?

São quantidades diferentes, e elas podem andar em sentidos opostos sem nenhuma
contradição. É aí que mora o dia.

---

### 🧮 O Modelo

Quatro camadas. A primeira mede, as duas do meio reconciliam, e a última mostra onde o
modelo falha.

1. **Posterior de Jeffreys** sobre os conflitos filmados, para dar intervalo ao número
   observado em vez de tratá-lo como certeza.
2. **As duas medidas ao mesmo tempo.** Traduzindo o g da meta-análise em queda de
   probabilidade individual por um deslocamento probit (hipótese minha, declarada) e
   agregando com P(alguém ajuda) = 1 − (1 − q)^m, sob independência.
3. **O limiar.** Quão baixa a chance de cada um precisaria ficar, num grupo de dez,
   para o grupo socorrer menos do que uma pessoa sozinha.
4. **A fenda do modelo.** O que a premissa de independência prevê, calibrada nos 3,76
   ajudantes por conflito, contra o que foi de fato observado, e onde a premissa quebra.

**Fontes:**
- Philpot, R., Liebst, L. S., Levine, M., Bernasco, W. & Lindegaard, M. R. (2019).
  "Would I Be Helped? Cross-National CCTV Footage Shows That Intervention Is the Norm in
  Public Conflicts". *American Psychologist* 75(1), 66-75. **219 conflitos públicos
  reais** filmados por câmeras de vigilância em Amsterdã (63), Cidade do Cabo (61) e
  Lancaster (95). Intervenção em **90,9%** deles (199 de 219; corpus inicial de 1.225 vídeos, triado
  por critérios de inclusão); média de **3,76 pessoas** intervindo por conflito (DP 3,01);
  cada espectador a mais esteve **associado** a maior chance de intervenção, razão de
  chances 1,10, IC 95% [1,03; 1,18], p = 0,008. Desenho observacional: associação, não
  causa.
- Fischer, P., Krueger, J. I., Greitemeyer, T., Vogrincic, C., Kastenmüller, A., Frey,
  D., Heene, M., Wicher, M. & Kainbacher, M. (2011). "The Bystander-Effect: A
  Meta-Analytic Review on Bystander Intervention in Dangerous and Non-Dangerous
  Emergencies". *Psychological Bulletin* 137(4), 517-537. **105 tamanhos de efeito
  independentes, mais de 7.700 participantes**, efeito geral **g = −0,35**, IC 95% [−0,40; −0,29]
  (efeitos fixos; −0,33 [−0,45; −0,22] em efeitos aleatórios), atenuado quando a
  situação é percebida como perigosa.
- Manning, R., Levine, M. & Collins, A. (2007). "The Kitty Genovese Murder and the
  Social Psychology of Helping: The Parable of the 38 Witnesses". *American
  Psychologist* 62(6), 555-562. Reconstrução do caso a partir das transcrições do
  julgamento, de documentos legais e de registros históricos.
- Stalder, D. R. (2008). "Revisiting the issue of safety in numbers: The likelihood of
  receiving help from a group". *Social Influence* 3(1), 24-33. Separa a inibição
  individual da probabilidade de a vítima receber ajuda de alguém do grupo. É a mesma
  distinção que este dia faz; entra como precedente, sem número usado no modelo.
- Darley, J. M. & Latané, B. (1968). Origem experimental do conceito. Entra como
  história, sem número usado no modelo.

**Nota metodológica:** o fator ×0,80 não se aplica. Os 90,9% vêm de codificação de vídeo
por observadores, não de autorrelato, e o g de 2011 é tamanho de efeito de meta-análise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

SEMENTE = 42

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def mil(n):
    return f"{int(n):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS DA LITERATURA ---

# Philpot et al. (2019), American Psychologist 75(1), 66-75
N_CONFLITOS = 219
CIDADES = {'Amsterdã': 63, 'Cidade do Cabo': 61, 'Lancaster': 95}
P_INTERVENCAO = 0.909
AJUDANTES_MEDIA, AJUDANTES_DP = 3.76, 3.01
OR_POR_ESPECTADOR = 1.1          # IC 95% [1,03; 1,18], p = 0,008 (associação)

# Fischer et al. (2011), Psychological Bulletin 137(4), 517-537
G_FISCHER = 0.35                 # queda individual, em unidades de g
K_EFEITOS, N_FISCHER = 105, 7700

aplica_fator_080 = False         # codificação de vídeo e meta-análise de efeitos

k_ajudados = round(N_CONFLITOS * P_INTERVENCAO)

print("=" * 70)
print("  DADOS DA LITERATURA")
print("=" * 70)
print(f"\n  PHILPOT 2019 · o que as câmeras de rua mostraram")
print(f"  {N_CONFLITOS} conflitos públicos reais: " +
      " · ".join(f"{c} {n}" for c, n in CIDADES.items()))
print(f"  → alguém interveio em {vir(P_INTERVENCAO*100)}% deles "
      f"({k_ajudados} de {N_CONFLITOS})")
print(f"  → média de {vir(AJUDANTES_MEDIA, 2)} pessoas intervindo por conflito "
      f"(DP {vir(AJUDANTES_DP, 2)})")
print(f"  → cada espectador a mais está ASSOCIADO a maior chance de ajuda: "
      f"razão de chances {vir(OR_POR_ESPECTADOR, 1)} [1,03; 1,18], p = 0,008")
print(f"\n  FISCHER 2011 · o efeito espectador clássico, medido")
print(f"  {K_EFEITOS} tamanhos de efeito independentes, mais de {mil(N_FISCHER)} pessoas")
print(f"  → efeito geral g = −{vir(G_FISCHER, 2)}, e ele é ATENUADO quando a situação")
print(f"     é percebida como perigosa")
print(f"\n  Os dois resultados não se contradizem. Eles medem coisas diferentes,")
print(f"  e o modelo abaixo mostra como as duas podem ser verdade ao mesmo tempo.")
print(f"\n  ×0,80: não se aplica.")
print("=" * 70)

In [ ]:
# --- O MODELO ---

# ---------------------------------------------------------------------------
# CAMADA 1 — posterior de Jeffreys sobre os conflitos filmados
a, b = k_ajudados + 0.5, N_CONFLITOS - k_ajudados + 0.5
post = stats.beta(a, b)
ic = post.ppf([0.025, 0.975])

print("=" * 70)
print("  CAMADA 1 · QUANTO VALE O 90,9%")
print("=" * 70)
print(f"\n  Beta({vir(a,1)} ; {vir(b,1)}) sobre {k_ajudados} de {N_CONFLITOS} conflitos")
print(f"  → observado {vir(k_ajudados/N_CONFLITOS*100)}%  ·  posterior "
      f"{vir(post.mean()*100)}%   IC 95% [{vir(ic[0]*100)}%, {vir(ic[1]*100)}%]")
print(f"  → P(acima de 80%)  = {vir(post.sf(0.80)*100, 1)}%")
print(f"  → P(acima de 2/3)  = {vir(post.sf(2/3)*100, 1)}%")
print(f"  A crença popular diz que ninguém ajuda. O intervalo inteiro fica acima de "
      f"{vir(ic[0]*100)}%.")

# ---------------------------------------------------------------------------
# CAMADA 2 — as duas medidas ao mesmo tempo
# Tradução do g num deslocamento probit da probabilidade individual.
def individual_com_plateia(p_sozinho, g=G_FISCHER):
    return stats.norm.cdf(stats.norm.ppf(p_sozinho) - g)

def alguem_ajuda(q, m):
    return 1 - (1 - q) ** m

BASES = [0.50, 0.65, 0.75]
GRUPOS = [1, 2, 3, 5, 10]

print("\n" + "=" * 70)
print("  CAMADA 2 · O INDIVÍDUO CAI E O GRUPO SOBE, AO MESMO TEMPO")
print("=" * 70)
print(f"\n  O g de Fischer (−{vir(G_FISCHER,2)}) traduzido num deslocamento probit da")
print(f"  propensão individual (hipótese minha), aplicado a três hipóteses de quanto")
print(f"  alguém sozinho ajudaria:\n")
print("  sozinho | com plateia | queda  |" + "".join(f"  grupo de {m:>2}" for m in GRUPOS))
print("  " + "-" * 76)
for p1 in BASES:
    q = individual_com_plateia(p1)
    linha = "".join(f"  {vir(alguem_ajuda(q, m)*100):>9}%" for m in GRUPOS)
    print(f"   {vir(p1*100):>5}% |   {vir(q*100):>6}%    | {vir((p1-q)*100)} pp |{linha}")
print("  " + "-" * 76)
print("  A coluna 'com plateia' é sempre menor que 'sozinho': o efeito espectador")
print("  existe. E as colunas da direita sobem mesmo assim.")

# Robustez da tradução: a conversão logit (ln OR = g·π/√3) é a outra convenção usual.
def individual_logit(p_sozinho, g=G_FISCHER):
    odds = p_sozinho / (1 - p_sozinho) * np.exp(-g * np.pi / np.sqrt(3))
    return odds / (1 + odds)
print(f"\n  Robustez: na base de 65%, o probit dá {vir(individual_com_plateia(0.65)*100)}% e a")
print(f"  conversão logit dá {vir(individual_logit(0.65)*100)}%. A escolha da tradução")
print(f"  muda o número em ~2 pontos e não muda a conclusão.")

# ---------------------------------------------------------------------------
# CAMADA 3 — o limiar
print("\n" + "=" * 70)
print("  CAMADA 3 · QUANTO A CHANCE INDIVIDUAL PRECISARIA CAIR")
print("=" * 70)
print("\n  Quão baixa a chance de cada um teria que ficar, num grupo de 10 pessoas")
print("  independentes, para o grupo ajudar MENOS que uma pessoa sozinha?")
print("  (parametrizado como decaimento constante por pessoa; equivale a resolver")
print("  1 − (1 − q10)^10 = q1)\n")
limiares = []
for q1 in [0.30, 0.50, 0.70]:
    f = lambda lam: alguem_ajuda(q1*np.exp(-lam*9), 10) - q1
    lam = optimize.brentq(f, 1e-9, 5)
    q10 = q1 * np.exp(-lam * 9)
    limiares.append((q1, q10))
    print(f"  Sozinho ajuda em {vir(q1*100, 0)}%: o grupo de 10 só fica pior se a chance")
    print(f"  de CADA UM despencar para {vir(q10*100)}%, uma queda de "
          f"{vir((1-q10/q1)*100, 0)}%.")
print(f"\n  A queda simulada a partir do g de Fischer fica perto de 14 pontos")
print(f"  (13,6 na base de 65%). O limiar pede quedas acima de 80%. Não está nem perto.")

# ---------------------------------------------------------------------------
# CAMADA 4 — a fenda do modelo
print("\n" + "=" * 70)
print("  CAMADA 4 · ONDE O MEU MODELO ERRA, E O QUE ISSO DIZ")
print("=" * 70)
print(f"\n  Se cada espectador decidisse por conta própria, sem olhar para os outros,")
print(f"  o q implícito nos {vir(AJUDANTES_MEDIA,2)} ajudantes por conflito daria:\n")
print("  presentes | cada um ajuda | P(alguém ajuda), com independência")
print("  " + "-" * 60)
fenda = []
for m in [4, 6, 8, 10, 15]:
    q = AJUDANTES_MEDIA / m
    p = alguem_ajuda(min(q, 1.0), m)
    fenda.append((m, q, p))
    print(f"     {m:>3}    |    {vir(q*100):>6}%    |            {vir(p*100)}%")
print("  " + "-" * 60)
print(f"  Observado por Philpot: {vir(P_INTERVENCAO*100)}%.")
print(f"\n  O modelo prevê ALTO DEMAIS, sempre acima de {vir(min(p for _,_,p in fenda)*100)}%.")
print(f"  A premissa de independência é generosa: na rua as pessoas olham umas para as")
print(f"  outras, hesitam juntas e decidem juntas. A fenda entre o previsto e o")
print(f"  observado mostra onde essa premissa quebra. Ela mistura dependência entre")
print(f"  decisões, diferenças entre conflitos e a calibração aproximada de q, então")
print(f"  NÃO é uma medida do efeito espectador.")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 — a posterior contra a crença
fig1, ax1 = plt.subplots(figsize=(12, 8))
x = np.linspace(0.78, 1.0, 1200)
y = post.pdf(x)
ax1.plot(x * 100, y, color=VERDE, linewidth=3, zorder=4)
ax1.fill_between(x * 100, y, color=VERDE, alpha=0.18, zorder=2)
ax1.axvspan(78, ic[0] * 100, color=CINZA, alpha=0.06, zorder=1)
for v, cor, txt in [(ic[0], DOURADO, f'{vir(ic[0]*100)}%'),
                    (ic[1], DOURADO, f'{vir(ic[1]*100)}%')]:
    ax1.axvline(v * 100, color=cor, linestyle='--', linewidth=2, zorder=3)
p_obs = k_ajudados / N_CONFLITOS
ax1.axvline(p_obs * 100, color=VERMELHO, linewidth=2.6, zorder=5)
ax1.text(p_obs * 100, max(y) * 1.03, f'{vir(p_obs*100)}%', ha='center',
         fontsize=15, fontweight='bold', color=VERMELHO)
ax1.text(ic[0] * 100 - 0.4, max(y) * 0.55, f'IC 95%\n[{vir(ic[0]*100)}% · {vir(ic[1]*100)}%]',
         ha='right', fontsize=11.5, color=DOURADO)
ax1.set_xlabel('Conflitos em que alguém interveio (%)', fontsize=12)
ax1.set_ylabel('Densidade da posterior', fontsize=12)
ax1.set_title('Em nove de cada dez conflitos filmados, alguém intervém\n'
              f'{k_ajudados} de {N_CONFLITOS} conflitos filmados em três países · '
              'posterior de Jeffreys', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, 'Fonte: Philpot et al., 2019, American Psychologist 75(1) '
            '| #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-110-grafico-01-nove-em-dez.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 — as duas medidas ao mesmo tempo
fig2, (axA, axB) = plt.subplots(1, 2, figsize=(14, 7))
ms = np.arange(1, 13)
p1 = 0.65
q = individual_com_plateia(p1)

axA.plot(ms, [p1 * 100] * len(ms), linestyle=':', linewidth=2.4, color=CINZA,
         label=f'se estivesse sozinho: {vir(p1*100, 0)}%')
axA.plot(ms, [q * 100] * len(ms), linewidth=3, color=VERMELHO,
         label=f'com plateia: {vir(q*100)}%')
axA.fill_between(ms, q * 100, p1 * 100, color=VERMELHO, alpha=0.12)
axA.annotate(f'o g de Fischer,\ntraduzido:\n{vir((p1-q)*100)} pontos',
             xy=(7, (p1 + q) / 2 * 100), xytext=(7, 30), ha='center', fontsize=11,
             color=VERMELHO, arrowprops=dict(arrowstyle='->', color=VERMELHO, lw=1.5))
axA.set_ylim(0, 100)
axA.set_xlabel('Pessoas presentes', fontsize=12)
axA.set_ylabel('Chance de UMA pessoa específica ajudar (%)', fontsize=12)
axA.set_title('A pergunta do laboratório\nO efeito espectador é real', fontsize=13, pad=14)
axA.legend(frameon=False, fontsize=11, loc='upper right')

axB.plot(ms, [alguem_ajuda(q, m) * 100 for m in ms], marker='o', markersize=7,
         linewidth=3, color=VERDE, zorder=3)
axB.axhline(P_INTERVENCAO * 100, color=DOURADO, linestyle='--', linewidth=2, zorder=2)
axB.text(11.8, P_INTERVENCAO * 100 - 8, f'o observado na rua: {vir(P_INTERVENCAO*100)}%',
         fontsize=11.5, color='#8a6d2a', ha='right')
axB.set_ylim(0, 105)
axB.set_xlabel('Pessoas presentes', fontsize=12)
axB.set_ylabel('Chance de ALGUÉM ajudar (%)', fontsize=12)
axB.set_title('A pergunta da calçada\nE mesmo assim a ajuda sobe', fontsize=13, pad=14)

fig2.suptitle('As duas coisas são verdade ao mesmo tempo', fontsize=15, y=0.99)
plt.figtext(0.5, 0.005, 'Fischer et al. 2011 traduzido em probabilidade · agregação '
            'P(alguém) = 1 − (1 − q) elevado a m · conta própria | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-110-grafico-02-duas-medidas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 — o limiar
fig3, ax3 = plt.subplots(figsize=(12, 8))
rot = [f'{vir(q1*100, 0)}%' for q1, _ in limiares]
xs = np.arange(len(limiares))
ax3.bar(xs - 0.2, [q1 * 100 for q1, _ in limiares], 0.38, color=VERDE, alpha=0.88,
        label='chance de ajudar sozinho', zorder=2)
ax3.bar(xs + 0.2, [q10 * 100 for _, q10 in limiares], 0.38, color=VERMELHO, alpha=0.88,
        label='onde ela precisaria chegar, num grupo de 10', zorder=2)
for i, (q1, q10) in enumerate(limiares):
    ax3.text(i - 0.2, q1 * 100 + 1.5, f'{vir(q1*100, 0)}%', ha='center',
             fontsize=12, fontweight='bold')
    ax3.text(i + 0.2, q10 * 100 + 1.5, f'{vir(q10*100)}%', ha='center',
             fontsize=12, fontweight='bold', color=VERMELHO)
    ax3.text(i, q1 * 100 + 9, f'queda de {vir((1-q10/q1)*100, 0)}%', ha='center',
             fontsize=11.5, color=CINZA)
ax3.set_xticks(xs); ax3.set_xticklabels(rot, fontsize=12)
ax3.set_xlabel('Quanto uma pessoa sozinha ajudaria', fontsize=12)
ax3.set_ylabel('Chance individual de ajudar (%)', fontsize=12)
ax3.set_ylim(0, 85)
ax3.set_title('O que precisaria acontecer para a multidão ser ruim\n'
              'A queda simulada a partir da literatura é de cerca de 14 pontos. '
              'O limiar pede mais de 80%.', fontsize=14, pad=16)
ax3.legend(frameon=False, fontsize=11.5)
plt.figtext(0.5, 0.01, 'Conta própria sobre P(alguém) = 1 − (1 − q) elevado a m '
            '| #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-110-grafico-03-o-limiar.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 — a fenda do modelo
fig4, ax4 = plt.subplots(figsize=(12, 8))
mm = [m for m, _, _ in fenda]
pp = [p * 100 for _, _, p in fenda]
ax4.plot(mm, pp, marker='o', markersize=9, linewidth=3, color=VERDE, zorder=3,
         label='o que o modelo prevê, se cada um decidisse sozinho')
ax4.axhline(P_INTERVENCAO * 100, color=VERMELHO, linestyle='--', linewidth=2.6, zorder=2,
            label=f'o que as câmeras mostraram: {vir(P_INTERVENCAO*100)}%')
ax4.fill_between(mm, P_INTERVENCAO * 100, pp, color=DOURADO, alpha=0.20, zorder=1)
ax4.annotate('a fenda é onde a premissa\nde independência quebra',
             xy=(8, (P_INTERVENCAO * 100 + pp[2]) / 2), xytext=(9.5, 93.5),
             fontsize=12, color='#8a6d2a', ha='center',
             arrowprops=dict(arrowstyle='->', color=DOURADO, lw=1.8))
ax4.set_ylim(88, 101)
ax4.set_xlabel('Pessoas presentes no conflito', fontsize=12)
ax4.set_ylabel('P(alguém ajuda) %', fontsize=12)
ax4.set_title('Onde o meu modelo erra, e por quê\n'
              'A independência prevê alto demais: na rua ninguém decide sozinho',
              fontsize=14, pad=16)
ax4.legend(frameon=False, fontsize=11.5, loc='lower right')
plt.figtext(0.5, 0.01, 'Conta própria calibrada nos 3,76 ajudantes por conflito de '
            'Philpot et al. 2019 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-110-grafico-04-a-fenda.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**Em 9 de cada 10 conflitos filmados, alguém faz alguma coisa.**

São 219 conflitos reais, filmados por câmeras de vigilância em Amsterdã, na Cidade do
Cabo e em Lancaster. Alguém interveio em 90,9% deles, e o intervalo de 95% dessa
estimativa (posterior de Jeffreys) vai de 86,5% a 94,1%. O pior caso do intervalo ainda
é quase nove em dez.

E não foi uma pessoa heroica por conflito. Foi uma média de **3,76 pessoas**.

O que derruba a crença de vez é a direção: **quanto mais espectadores havia, maior a
chance de alguém intervir**, razão de chances de 1,1 por pessoa. A curva não desce. Sobe.

Agora, o efeito espectador clássico não é mentira. Ele está medido em 105 tamanhos de
efeito e mais de 7.700 pessoas, com g de −0,35. A chance de **uma pessoa específica**
agir cai mesmo quando há gente em volta. Na minha simulação, traduzindo esse g numa
escala de probabilidade, a queda fica perto de **14 pontos percentuais**.

As duas coisas são verdade ao mesmo tempo, e é aritmética.

Se alguém sozinho ajudaria em 65% das vezes, a plateia derruba isso para 51,4%. Mas
basta haver duas pessoas para a chance de **alguém** ajudar subir para 76,4%. Com cinco,
97,3%.

Fiz a conta ao contrário para saber o tamanho do que seria preciso. Para um grupo de dez
pessoas socorrer menos do que uma pessoa sozinha, a chance de cada uma precisaria cair
de 65% para menos de 10%, uma queda de **85%**. A simulação com a meta-análise dá 14
pontos.

E o meu próprio modelo erra, na direção que ensina alguma coisa. Se cada espectador
decidisse por conta própria, sem olhar para os outros, a conta calibrada nos 3,76
ajudantes por conflito previa mais de 98% de conflitos socorridos. O observado foi
90,9%. **A diferença mostra onde a premissa de independência quebra:** na rua, as
pessoas se olham, hesitam juntas e decidem juntas. Stalder já tinha separado as duas
perguntas em 2008 e chegado ao mesmo lugar: cada um pode hesitar mais, e a vítima pode
receber mais ajuda.

Cada um hesita. O grupo não abandona.

A pergunta que fica:

*Quantas vezes você não pediu ajuda porque tinha certeza de que ninguém viria?*

---

### ⚠️ Limitações do Modelo

- **O estudo mede intervenção, não resultado.** Intervir inclui gestos pequenos, como
  falar, apartar ou tocar no ombro de quem está agressivo. Não significa que o conflito
  terminou bem, nem que quem interveio saiu ileso.
- **São conflitos públicos e gravados.** Câmeras de vigilância ficam em centros de
  cidade, com luz e movimento. Emergência dentro de casa, em estrada vazia ou de
  madrugada é outra população, e este número não fala por ela.
- **A tradução do g em probabilidade é conta minha, não resultado da meta-análise.**
  Uso um deslocamento probit sobre três hipóteses de quanto alguém ajudaria sozinho,
  porque a meta-análise publica tamanho de efeito, não taxa absoluta. Os "14 pontos"
  são desta simulação, na base de 65%; a literatura mede g = −0,35. A outra convenção
  usual (logit, ln OR = g·π/√3) dá 49,6% em vez de 51,4%. As três hipóteses e as duas
  conversões estão no console para mostrar que a conclusão não depende da escolha.
- **A premissa de independência é falsa, e eu uso assim mesmo.** A Camada 4 calibra a
  chance de cada um nos 3,76 ajudantes médios por conflito (q = 3,76/m) e roda grupos
  de 4 a 15 pessoas; com os 16,3 espectadores médios do estudo, a conta chega perto de
  100%. A fenda até os 90,9% mostra onde a premissa quebra, mas mistura dependência
  entre decisões, heterogeneidade entre conflitos e a calibração aproximada de q. Não é
  uma medida do efeito espectador, e os 98% não são previsão.
- **O 90,9% é de conflitos públicos filmados, não de qualquer emergência.** O corpus
  partiu de 1.225 vídeos e chegou a 219 por critérios de inclusão; "intervir" inclui
  gesto, toque, afastar ou consolar. E o estudo não compara rua com e sem câmera, então
  não responde se ser filmado muda o comportamento.
- **Os três países não são o mundo.** Amsterdã, Cidade do Cabo e Lancaster foram
  escolhidas por disponibilidade de imagens. Nenhum dado brasileiro entrou.
- **A atenuação em situações perigosas é achado dos autores da meta-análise, não teste
  meu.** Ela aparece aqui como contexto, e ajuda a explicar por que a rua se comporta
  diferente do laboratório.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*